# Bibliometric Records, Network Data, and Citation Metrics from Web of Science and Related Sources Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and processing the "Bibliometric Records, Network Data, and Citation Metrics from Web of Science and Related Sources" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.x5rc-x00h/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This section imports essential libraries, sets the dataset URL, and loads metadata for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.x5rc-x00h/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")
print(f"Dataset Identifier: {metadata.identifier}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their unique IDs (`@id`). This section lists the record sets, their associated fields and columns, using the Croissant schema structure.

**Note:** All entities are referenced by their `@id` as per Croissant schema best practices.

In [ ]:
# Print all record sets and their associated fields and columns
record_sets = metadata.recordSet
if not record_sets:
    print("No record sets found in dataset metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet '@id': {rs['@id']}")
        print(f"    Name: {rs.get('name', 'N/A')}")
        
        fields = rs.get('field', [])
        columns = rs.get('column', [])

        if fields:
            print("    Fields:")
            for field in fields:
                print(f"        - {field['@id']} (Name: {field.get('name', 'N/A')}, DataType: {field.get('dataType', 'N/A')})")
        else:
            print("    No fields listed.")

        if columns:
            print("    Columns:")
            for col in columns:
                print(f"        - {col['@id']} (Name: {col.get('name', 'N/A')}, DataType: {col.get('dataType', 'N/A')})")
        else:
            print("    No columns listed.")

        print()
# For the purposes of demonstration, let's extract a sample record set if available
sample_record_set_id = None
if record_sets:
    sample_record_set_id = record_sets[0]['@id']

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis.

Each record set is referenced by its `@id`. Columns and fields are also referenced only by their `@id`.

In [ ]:
# Extract data from available record sets
dataframes = {}

# Collect list of record set '@id's
record_sets_ids = [rs['@id'] for rs in record_sets] if record_sets else []

# Load each record set into a pandas DataFrame
for rs_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet {rs_id}")
    except Exception as e:
        print(f"Error loading records for {rs_id}: {e}")

# Display columns for the first loaded record set
if sample_record_set_id and sample_record_set_id in dataframes:
    print(f"Columns for RecordSet {sample_record_set_id}: {dataframes[sample_record_set_id].columns.tolist()}")
    display(dataframes[sample_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering records, normalizing numeric fields, and grouping by key attributes.

For demonstration, these steps use the first available numeric field and group-by field, referenced by their `@id`. Update the variables if you want to explore other fields.


In [ ]:
# EDA: Filtering, normalization, grouping

# Example: use the first record set and search its columns for a numeric field
active_rs_id = sample_record_set_id
df = dataframes.get(active_rs_id, pd.DataFrame())

numeric_field_id = None
group_field_id = None

if not df.empty:
    # Identify a numeric column (float or integer)
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    # Identify a groupable field (categorical/object type)
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break

    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped '{numeric_field_id}' mean by '{group_field_id}':")
            display(grouped_df.head())
    else:
        print("No numeric field found in the dataframe.")
else:
    print("No data for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields using matplotlib for basic plots.

Update the variables for fields to visualize as needed.

In [ ]:
# Basic visualization
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field_id and group_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in RecordSet {active_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"'{numeric_field_id}' by '{group_field_id}' in RecordSet {active_rs_id}")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("No fields found for visualization or data empty.")

## 6. Conclusion
This notebook demonstrated loading, overview, extraction, simple EDA, and visualization of the Web of Science dataset using Croissant metadata via the `mlcroissant` library.

Key observations:
- Dataset structure (record sets, fields) allows targeted extraction and manipulation.
- Referencing by `@id` ensures reproducibility and clarity.
- Analysis and visualization can be extended by selecting specific fields via their `@id`s.

Please refer to the Croissant schema and `mlcroissant` documentation for further customization and deeper analyses.